# 02 - Evaluation baseline

Entrainement du modele de reference puis calcul des metriques de performance et fairness.

In [1]:
from pathlib import Path
import os
import sys
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

for path in ['data/processed', 'models', 'outputs', 'reports', 'reports/figures']:
    (PROJECT_ROOT / path).mkdir(parents=True, exist_ok=True)

In [2]:
def sanitize(obj):
    import numpy as np
    import math
    if isinstance(obj, dict):
        return {str(k): sanitize(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [sanitize(v) for v in obj]
    if isinstance(obj, tuple):
        return [sanitize(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return sanitize(obj.tolist())
    if isinstance(obj, np.generic):
        return sanitize(obj.item())
    if isinstance(obj, float) and not math.isfinite(obj):
        return None
    return obj

In [3]:
import json
import pickle
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LogisticRegression

from src.data_processing import DataProcessor, generate_sample_data
from src.metrics import FairnessMetrics, PerformanceMetrics

DATA_PATH = PROJECT_ROOT / 'data/processed/dataset.csv'
if not DATA_PATH.exists():
    generate_sample_data(n_samples=5000, random_state=42).to_csv(DATA_PATH, index=False)

df = pd.read_csv(DATA_PATH)
processor = DataProcessor(protected_attributes=['gender', 'race'], label_name='label')
df_encoded = processor.encode_categorical(df)
X_train, X_test, y_train, y_test = processor.split_data(df_encoded, test_size=0.2, random_state=42)

protected_attrs = ['gender', 'race']
X_train_model = X_train.drop(columns=protected_attrs)
X_test_model = X_test.drop(columns=protected_attrs)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_model, y_train)

with open(PROJECT_ROOT / 'models/baseline_model.pkl', 'wb') as f:
    pickle.dump(model, f)

y_pred = model.predict(X_test_model)
y_scores = model.predict_proba(X_test_model)[:, 1]
performance = PerformanceMetrics.compute_metrics(y_test, y_pred, y_scores)

fairness = {}
for attr in protected_attrs:
    values = X_test[attr].unique()
    fm = FairnessMetrics(attr, privileged_value=values[0], unprivileged_value=values[1] if len(values) > 1 else None)
    fairness[attr] = fm.compute_all_metrics(y_test, y_pred, X_test[attr], y_scores)

results = {'performance': performance, 'fairness': fairness}
with open(PROJECT_ROOT / 'outputs/baseline_results.json', 'w', encoding='utf-8') as f:
    json.dump(sanitize(results), f, indent=2, ensure_ascii=False)

print('Performance baseline')
for key in ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']:
    print(f'{key}: {performance[key]:.3f}')

display(pd.DataFrame(fairness).T[['demographic_parity_difference', 'disparate_impact', 'average_odds_difference']])

Performance baseline
accuracy: 0.658
precision: 0.662
recall: 0.965
f1_score: 0.785
roc_auc: 0.615


,demographic_parity_difference,disparate_impact,average_odds_difference
gender,-0.037761,1.040942,-0.034308
race,0.034575,0.96355,0.021981


In [4]:
summary = pd.DataFrame(fairness).T[['demographic_parity_difference', 'average_odds_difference']]
fig, ax = plt.subplots(figsize=(7, 4))
summary.plot(kind='bar', ax=ax)
ax.set_title('Ecarts de fairness - baseline')
ax.set_ylabel('Ecart absolu')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
fig.savefig(PROJECT_ROOT / 'reports/figures/baseline_fairness.png', dpi=150, bbox_inches='tight')
plt.close(fig)